# Biblical Qwen 2.5 14B Fine-Tuning with Unsloth (4-bit QLoRA)

**Base Model:** Qwen 2.5 14B Instruct

**Dataset:** Augmentoolkit-generated Biblical persona (first-person apostolic responses)

**Training Hardware:** NVIDIA DGX Spark (128GB unified memory)

**Deployment Target:** A5000 GPU server via vLLM (24GB VRAM — 14B 4-bit ~8GB fits comfortably)

**Chat Template:** `<|im_start|>role\ncontent<|im_end|>` (ChatML)

## Step 1: Configuration

All paths and variables for easy configuration.

In [1]:
# ============================================================================
# CONFIGURATION - All variables for easy setup
# ============================================================================
# Training: DGX Spark (128GB unified memory)
# Deployment: A5000 GPU server (24GB VRAM) via vLLM (14B 4-bit ~8GB)

# =========================== MODEL CONFIGURATION ===========================
BASE_LLM = "unsloth/Qwen2.5-14B-Instruct-bnb-4bit"
MODEL_NAME_BASE = "biblical_qwen25_14b_instruct_unsloth_4bit"

# =========================== INPUT DATA ===========================
INPUT_DATA_PATH = "/home/spark/projects/augmentoolkit/outputs/biblical_persona_dataset"

# =========================== OUTPUT DIRECTORIES ===========================
OUTPUT_BASE_DIR = f"./output/{MODEL_NAME_BASE}"
OUTPUT_DIR_ADAPTERS = f"{OUTPUT_BASE_DIR}/train"

# =========================== TRAINING HYPERPARAMETERS ===========================
MAX_SEQ_LENGTH = 2048
BATCH_SIZE = 1
GRAD_ACCUM = 8
LEARNING_RATE = 1e-4
TARGET_EPOCHS = 1

# =========================== LoRA CONFIGURATION ===========================
# r=8, alpha=16: conservative, preserves base model behavior
# Increase r to 16-64 for more aggressive adaptation
LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0

# Target modules:
#   Conservative (persona): ["q_proj", "v_proj"]
#   Balanced:               ["q_proj", "k_proj", "v_proj", "o_proj"]
#   Aggressive (knowledge): ["q_proj", "k_proj", "v_proj", "o_proj",
#                            "gate_proj", "up_proj", "down_proj"]
LORA_TARGET_MODULES = ["q_proj", "v_proj"]

# =========================== VLLM DEPLOYMENT ===========================
VLLM_MODELS_DIR = "/home/spark/projects/stoic/output/vllm"

# =========================== INFERENCE TEST ===========================
TEST_PROMPT = "I am struggling with forgiveness. What does Scripture teach about forgiving others?"

# ============================================================================
print("\u2713 Configuration loaded (14B 4-bit QLoRA version)")
print(f"  Base model: {BASE_LLM}")
print(f"  Model name: {MODEL_NAME_BASE}")
print(f"  Input data: {INPUT_DATA_PATH}")
print(f"  Output base: {OUTPUT_BASE_DIR}")
print(f"  LoRA config: r={LORA_R}, alpha={LORA_ALPHA}, targets={LORA_TARGET_MODULES}")
print(f"  Training: batch={BATCH_SIZE}, grad_accum={GRAD_ACCUM}, lr={LEARNING_RATE}")
print(f"  Training precision: 4-bit QLoRA")
print(f"  Deployment: vLLM on A5000 GPU server")

## 2. Environment Preparation

Install Unsloth and updated HuggingFace libraries.

In [2]:
# Install core packages from PyPI (much faster than git installs)
!pip install -q unsloth transformers trl peft accelerate datasets bitsandbytes

# Verify installations
import unsloth
import transformers
import trl
print(f"\u2713 Unsloth: {unsloth.__version__}")
print(f"\u2713 Transformers: {transformers.__version__}")
print(f"\u2713 TRL: {trl.__version__}")
print("Environment ready!")

## 3. Load Dataset & Format for Instruction Tuning

Load the Augmentoolkit-generated Biblical dataset (first-person apostolic responses from Paul, David, Peter, etc.).

In [3]:
from datasets import load_dataset, concatenate_datasets
import glob

# Load ALL subdirectories and ALL files
all_dirs = glob.glob(f"{INPUT_DATA_PATH}/*/")

print("LOADING ALL AUGMENTOOLKIT DATA")
print(f"Found {len(all_dirs)} subdirectories")

datasets = []
for dir_path in sorted(all_dirs):
    jsonl_files = glob.glob(f"{dir_path}/*.jsonl")
    for file_path in jsonl_files:
        try:
            ds = load_dataset("json", data_files=file_path, split="train")
            datasets.append(ds)
            print(f"  Loaded {len(ds)} from {dir_path.split('/')[-2]}/{file_path.split('/')[-1]}")
        except Exception as e:
            print(f"  Skipped {file_path.split('/')[-1]}: {e}")

dataset = concatenate_datasets(datasets)

print(f"\nTotal dataset loaded: {len(dataset)} examples")
print(f"Columns: {dataset.column_names}")
print(f"\n--- First Example (first 800 chars) ---")
import json
print(json.dumps(dataset[0], indent=2)[:800])

# Shuffle dataset for better training
dataset = dataset.shuffle(seed=42)
print(f"\nDataset shuffled and ready for training")

## 4. Load Model & Tokenizer with Unsloth (4-bit)

Load Qwen 2.5 14B Instruct model in **4-bit precision** for QLoRA training.

**Training on DGX Spark (128GB unified memory):**
- 14B in 4-bit fits comfortably (~8GB)
- Plenty of headroom for optimizer states and activations

**Deployment on A5000 (24GB VRAM):**
- 14B 4-bit ~8GB, fits within 24GB with room for KV-cache
- vLLM handles KV-cache management efficiently

In [4]:
from unsloth import FastLanguageModel
import torch

# Load model in 4-bit precision for QLoRA training
# Use device_map={"":  0} to force all layers onto GPU 0
# DGX Spark has 119.7GB \u2014 plenty for 14B 4-bit (~8GB)
# device_map="auto" incorrectly offloads some layers to CPU which BnB 4-bit rejects
model, tokenizer = FastLanguageModel.from_pretrained(
    BASE_LLM,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,           # Auto-detect (will use bfloat16 for compute)
    load_in_4bit=True,    # 4-bit quantization for QLoRA
    device_map={"":  0}    # Force all layers onto GPU 0 (119GB unified memory)
)

tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print(f"\u2713 Model loaded: {BASE_LLM}")
print(f"\u2713 Precision: 4-bit (QLoRA)")
print(f"\u2713 Tokenizer configured")
print(f"\u2713 Max sequence length: {MAX_SEQ_LENGTH}")

In [5]:
# Format dataset for Qwen 2.5 chat template (ChatML)
# Handle BOTH ShareGPT format (conversations) AND raw text format
def format_instruct(example):
    # If has conversations field AND it's not null, convert ShareGPT to chat template
    if example.get("conversations") is not None:
        messages = []
        for turn in example["conversations"]:
            # ShareGPT uses "from": "system"/"human"/"gpt"
            # Standard uses "role": "system"/"user"/"assistant"
            if turn["from"] == "system":
                messages.append({"role": "system", "content": turn["value"]})
            elif turn["from"] == "human":
                messages.append({"role": "user", "content": turn["value"]})
            elif turn["from"] == "gpt":
                messages.append({"role": "assistant", "content": turn["value"]})
        
        text = tokenizer.apply_chat_template(
            messages, 
            tokenize=False, 
            add_generation_prompt=False
        )
        return {"text": text}
    
    # Otherwise if it has a text field (raw text files), keep it as-is
    elif example.get("text") is not None and len(str(example["text"])) > 0:
        return {"text": str(example["text"])}
    
    # Skip malformed examples
    return {"text": ""}

# Format and keep only text column
dataset = dataset.map(format_instruct, remove_columns=dataset.column_names)

# Filter out empty examples
dataset = dataset.filter(lambda x: len(x["text"]) > 0)

print(f"\n--- Sample formatted text (first 500 chars) ---")
print(dataset[0]['text'][:500])
print(f"\n\u2713 Dataset formatted: {len(dataset)} examples")

## 5. Add LoRA Adapters

Configure LoRA for efficient fine-tuning. See configuration section for module explanations.

In [6]:
from peft import LoraConfig

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=LORA_TARGET_MODULES,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    max_seq_length=MAX_SEQ_LENGTH
)

print(f"\u2713 LoRA adapters added (r={LORA_R}, targets={LORA_TARGET_MODULES})")
print(f"\u2713 Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

## 6. Trainer Setup & Training

**PURE DOMAIN DATA with LIGHT TRAINING:**
- 100% authentic Biblical examples from epistles and psalms
- 1 epoch with low learning rate (1e-4) to gently teach persona
- 14B is already highly capable \u2014 minimal fine-tuning is sufficient
- This preserves base model capabilities while adding authentic voice

In [7]:
from trl import SFTTrainer
from transformers import TrainingArguments

# Calculate training steps
effective_batch_size = BATCH_SIZE * GRAD_ACCUM
steps_per_epoch = len(dataset) // effective_batch_size
max_steps = steps_per_epoch * TARGET_EPOCHS
warmup_steps = max(1, max_steps // 10)
save_steps = min(100, steps_per_epoch)  # Save every 100 steps or at epoch end

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        warmup_steps=warmup_steps,
        max_steps=max_steps,
        learning_rate=LEARNING_RATE,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=5,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=3407,
        output_dir=OUTPUT_DIR_ADAPTERS,
        report_to="none",
        save_strategy="steps",
        save_steps=save_steps,
    )
)

print("\u2713 Trainer configured for 14B 4-bit QLoRA training")
print(f"\u2713 Dataset size: {len(dataset)} conversations")
print(f"\u2713 Effective batch size: {effective_batch_size} (batch={BATCH_SIZE} \u00d7 grad_accum={GRAD_ACCUM})")
print(f"\u2713 Steps per epoch: {steps_per_epoch}")
print(f"\u2713 Total epochs: {TARGET_EPOCHS}")
print(f"\u2713 Total steps: {max_steps}")
print(f"\u2713 Warmup steps: {warmup_steps}")
print(f"\u2713 Save every: {save_steps} steps")
print(f"\u2713 Learning rate: {LEARNING_RATE}")

In [8]:
# Start training
trainer.train()

## 7. Save LoRA Adapters

Save the trained LoRA adapters separately. These can be loaded on any Qwen 2.5 14B model later.

**This is the primary output** - merging to full model is optional (Step 8).

In [ ]:
from pathlib import Path

# Save LoRA adapters (primary output)
LORA_OUTPUT_DIR = f"{OUTPUT_BASE_DIR}/lora_adapters"
Path(LORA_OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

print(f"Saving LoRA adapters to {LORA_OUTPUT_DIR}...")
model.save_pretrained(LORA_OUTPUT_DIR)
tokenizer.save_pretrained(LORA_OUTPUT_DIR)

print(f"\u2713 LoRA adapters saved!")
print(f"   Path: {LORA_OUTPUT_DIR}")
print(f"   Can be loaded on any Qwen 2.5 14B model with PEFT")